# FER2013 LiteCNN Colab Notebook (Baseline Reproduction + V2 Improvement)

This notebook is fully standalone for Google Colab (A100 recommended).

What this notebook does:
1. Reproduce Jasmine-style LiteCNN baseline first and save best checkpoint.
2. Evaluate against Jasmine reference metrics.
3. Optionally run multi-seed baseline experiments to measure variance.
4. Define and train an improved LiteCNN v2 (no transfer learning) and compare results.

Expected Colab A100 runtime (rough guide):
- Baseline 50 epochs: ~12 to 30 minutes
- One improved model run (50 to 70 epochs): ~15 to 40 minutes
- Multi-seed (3 seeds, full runs): ~45 to 150 minutes total

These are estimates and depend on dataloader workers, storage speed, and current Colab load.

## 1. Set Seeds and Reproducibility Controls

This section sets deterministic behavior where possible. Baseline seed defaults to 42 (to mirror Jasmine-style baseline runs), and later cells let you vary seeds.

In [ ]:
import os
import random
import time
import math
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

try:
    import tensorflow as tf
except Exception:
    tf = None


def set_seed(seed: int = 42, deterministic: bool = True):
    random.seed(seed)
    np.random.seed(seed)

    if tf is not None:
        try:
            tf.random.set_seed(seed)
        except Exception:
            pass

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


BASELINE_SEED = 42
set_seed(BASELINE_SEED, deterministic=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f"GPU count: {torch.cuda.device_count()}")

## 2. Verify FER2013 Dataset Path and Load Files from `/content/data`

Expected folder layout:

- `/content/data/fer2013/train/<emotion>/*.png`
- `/content/data/fer2013/test/<emotion>/*.png`

Emotion folder names must be:
`angry, disgust, fear, happy, sad, surprise, neutral`

In [ ]:
EMOTIONS = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".gif"}

DATA_ROOT = Path("/content/data/fer2013")
TRAIN_ROOT = DATA_ROOT / "train"
TEST_ROOT = DATA_ROOT / "test"


def count_images_by_class(root: Path):
    counts = {}
    total = 0
    for e in EMOTIONS:
        cls_dir = root / e
        n = 0
        if cls_dir.is_dir():
            for p in cls_dir.iterdir():
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                    n += 1
        counts[e] = n
        total += n
    return counts, total


print(f"DATA_ROOT: {DATA_ROOT}")
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        "Dataset folder not found. Please upload FER2013 into /content/data/fer2013 first."
    )

train_counts, train_total = count_images_by_class(TRAIN_ROOT)
test_counts, test_total = count_images_by_class(TEST_ROOT)

print("Train counts:")
print(train_counts)
print(f"Total train images: {train_total}")
print("\nTest counts:")
print(test_counts)
print(f"Total test images: {test_total}")

if train_total == 0 or test_total == 0:
    raise RuntimeError("Train/Test image counts are zero. Check folder structure under /content/data/fer2013.")

## 3. Preprocess Images and Labels

## 4. Create Train/Validation/Test Splits

This follows Jasmine-style folder loading with class-wise split from train into train/val using a fixed seed.

In [ ]:
def load_gray(path: Path):
    return np.asarray(Image.open(path).convert("L"), dtype=np.uint8)


def split_classwise(items, val_fraction=0.1, seed=42):
    if len(items) == 0:
        return [], []
    rng = np.random.default_rng(seed)
    idx = np.arange(len(items))
    rng.shuffle(idx)

    val_size = int(round(len(items) * val_fraction))
    if val_size == 0 and len(items) > 1:
        val_size = 1
    if val_size >= len(items):
        val_size = len(items) - 1

    val_idx = set(idx[:val_size])
    train_items = [x for i, x in enumerate(items) if i not in val_idx]
    val_items = [x for i, x in enumerate(items) if i in val_idx]
    return train_items, val_items


class FERFolderDataset(Dataset):
    def __init__(self, root: Path, split="train", transform=None, val_fraction=0.1, seed=42):
        self.root = root
        self.split = split.lower()
        self.transform = transform
        self.val_fraction = val_fraction
        self.seed = seed

        self.images = []
        self.labels = []

        train_root = root / "train"
        test_root = root / "test"

        if self.split == "train":
            self._load_train_or_val(train_root, use_val=False)
        elif self.split == "val":
            self._load_train_or_val(train_root, use_val=True)
        elif self.split == "test":
            self._load_all(test_root)
        else:
            raise ValueError(f"Unsupported split: {split}")

    def _collect_class_items(self, class_dir: Path):
        items = []
        if class_dir.is_dir():
            for p in sorted(class_dir.iterdir()):
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                    items.append(load_gray(p))
        return items

    def _load_train_or_val(self, train_root: Path, use_val: bool):
        for class_idx, emo in enumerate(EMOTIONS):
            class_items = self._collect_class_items(train_root / emo)
            train_items, val_items = split_classwise(
                class_items,
                val_fraction=self.val_fraction,
                seed=self.seed + class_idx,
            )
            selected = val_items if use_val else train_items
            self.images.extend(selected)
            self.labels.extend([class_idx] * len(selected))

    def _load_all(self, folder_root: Path):
        for class_idx, emo in enumerate(EMOTIONS):
            for img in self._collect_class_items(folder_root / emo):
                self.images.append(img)
                self.labels.append(class_idx)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx])
        label = int(self.labels[idx])
        if self.transform is not None:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)


train_tfms_baseline = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

test_tfms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])


def make_loaders(
    data_root=DATA_ROOT,
    batch_size=64,
    val_fraction=0.1,
    split_seed=42,
    train_tfms=None,
    test_tfms_local=None,
    num_workers=2,
    pin_memory=True,
):
    if train_tfms is None:
        train_tfms = train_tfms_baseline
    if test_tfms_local is None:
        test_tfms_local = test_tfms

    train_ds = FERFolderDataset(data_root, split="train", transform=train_tfms, val_fraction=val_fraction, seed=split_seed)
    val_ds = FERFolderDataset(data_root, split="val", transform=test_tfms_local, val_fraction=val_fraction, seed=split_seed)
    test_ds = FERFolderDataset(data_root, split="test", transform=test_tfms_local, val_fraction=val_fraction, seed=split_seed)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = make_loaders(batch_size=64, split_seed=BASELINE_SEED)
print(f"Train samples: {len(train_loader.dataset)}")
print(f"Val samples:   {len(val_loader.dataset)}")
print(f"Test samples:  {len(test_loader.dataset)}")

batch_x, batch_y = next(iter(train_loader))
print(f"Batch shape: {tuple(batch_x.shape)} | Labels shape: {tuple(batch_y.shape)}")

## 5. Reproduce Jasmine's Baseline Model

This is the same LiteCNN family idea used in your project code: depthwise separable convolutions + residual blocks + global average pooling classifier.

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size=3, stride=stride, padding=1,
            groups=in_channels, bias=False,
        )
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU6(inplace=True)

    def forward(self, x):
        x = self.relu(self.bn1(self.depthwise(x)))
        x = self.relu(self.bn2(self.pointwise(x)))
        return x


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = DepthwiseSeparableConv(in_channels, out_channels, stride)
        self.conv2 = DepthwiseSeparableConv(out_channels, out_channels, stride=1)
        self.relu = nn.ReLU6(inplace=True)
        self.skip = None
        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        residual = x if self.skip is None else self.skip(x)
        out = self.conv1(x)
        out = self.conv2(out)
        out = out + residual
        out = self.relu(out)
        return out


class LiteCNNBaseline(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.init_conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True),
        )
        self.block1 = self._make_layer(32, 48, 2, stride=2)
        self.block2 = self._make_layer(48, 96, 2, stride=2)
        self.block3 = self._make_layer(96, 192, 3, stride=2)
        self.block4 = self._make_layer(192, 384, 2, stride=2)
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(384, num_classes),
        )

    def _make_layer(self, in_channels, out_channels, num_blocks, stride):
        layers = [ResidualBlock(in_channels, out_channels, stride)]
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.init_conv(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.global_avg_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


baseline_model = LiteCNNBaseline(num_classes=7)
print(f"LiteCNNBaseline params: {count_parameters(baseline_model):,}")

## 6. Train Baseline and Save the Best Model

## 7. Evaluate Baseline Metrics Against Jasmine's Results

Baseline setup (matching Jasmine-style recipe):
- Adam, lr=1e-3, weight_decay=1e-4
- CosineAnnealingLR
- CrossEntropyLoss
- 50 epochs default
- Best checkpoint by validation accuracy

In [ ]:
@dataclass
class TrainConfig:
    epochs: int = 50
    batch_size: int = 64
    lr: float = 1e-3
    weight_decay: float = 1e-4
    val_fraction: float = 0.1
    split_seed: int = 42
    run_seed: int = 42
    num_workers: int = 2
    early_stopping_patience: int = 0  # 0 means disabled for strict Jasmine-style baseline


def evaluate_accuracy(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            total += y.size(0)
            correct += (pred == y).sum().item()
    return 100.0 * correct / total if total > 0 else 0.0


def train_model(model, train_loader, val_loader, cfg: TrainConfig, checkpoint_path: Path, class_weights=None, label_smoothing=0.0):
    set_seed(cfg.run_seed, deterministic=True)
    model = model.to(DEVICE)

    if class_weights is not None:
        class_weights = class_weights.to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=1e-4)

    history = {
        "epoch": [],
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": [],
    }

    best_val_acc = 0.0
    best_epoch = -1
    best_state = None
    stale_epochs = 0

    start_total = time.time()
    for epoch in range(cfg.epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)
            pred = logits.argmax(dim=1)
            total += y.size(0)
            correct += (pred == y).sum().item()

        train_loss = running_loss / max(total, 1)
        train_acc = 100.0 * correct / max(total, 1)

        model.eval()
        val_running_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits = model(x)
                loss = criterion(logits, y)
                val_running_loss += loss.item() * x.size(0)
                pred = logits.argmax(dim=1)
                val_total += y.size(0)
                val_correct += (pred == y).sum().item()

        val_loss = val_running_loss / max(val_total, 1)
        val_acc = 100.0 * val_correct / max(val_total, 1)

        scheduler.step()

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(optimizer.param_groups[0]["lr"])

        print(
            f"Epoch {epoch+1:03d}/{cfg.epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.2f}% | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.2f}%"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1

        if cfg.early_stopping_patience > 0 and stale_epochs >= cfg.early_stopping_patience:
            print(f"Early stopping triggered at epoch {epoch+1}.")
            break

    elapsed = time.time() - start_total

    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    if best_state is not None:
        torch.save(best_state, checkpoint_path)
        model.load_state_dict(best_state)

    test_info = {
        "best_val_acc": best_val_acc,
        "best_epoch": best_epoch,
        "elapsed_sec": elapsed,
        "checkpoint_path": str(checkpoint_path),
    }
    return model, history, test_info


def run_baseline_once(seed=42, epochs=50):
    cfg = TrainConfig(
        epochs=epochs,
        batch_size=64,
        lr=1e-3,
        weight_decay=1e-4,
        val_fraction=0.1,
        split_seed=seed,
        run_seed=seed,
        num_workers=2,
        early_stopping_patience=0,
    )

    train_loader, val_loader, test_loader = make_loaders(
        data_root=DATA_ROOT,
        batch_size=cfg.batch_size,
        val_fraction=cfg.val_fraction,
        split_seed=cfg.split_seed,
        train_tfms=train_tfms_baseline,
        test_tfms_local=test_tfms,
        num_workers=cfg.num_workers,
        pin_memory=True,
    )

    model = LiteCNNBaseline(num_classes=7)
    ckpt = Path("saved_models") / f"LiteCNN_baseline_seed{seed}.pth"

    model, history, info = train_model(model, train_loader, val_loader, cfg, ckpt)

    test_acc = evaluate_accuracy(model, test_loader, DEVICE)
    info["test_acc"] = test_acc

    print("\nBaseline summary")
    print(info)
    print(f"Test accuracy: {test_acc:.2f}%")

    hist_df = pd.DataFrame(history)
    return model, hist_df, info


RUN_BASELINE_NOW = True
BASELINE_EPOCHS = 50

if RUN_BASELINE_NOW:
    baseline_model, baseline_hist_df, baseline_info = run_baseline_once(seed=BASELINE_SEED, epochs=BASELINE_EPOCHS)

    jasmine_ref_val = 65.73
    jasmine_ref_test = 66.09

    print("\nComparison vs Jasmine reference")
    print(f"Your baseline best val: {baseline_info['best_val_acc']:.2f}% | Jasmine ref val: {jasmine_ref_val:.2f}%")
    print(f"Your baseline test:     {baseline_info['test_acc']:.2f}% | Jasmine ref test: {jasmine_ref_test:.2f}%")

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(baseline_hist_df["epoch"], baseline_hist_df["train_acc"], label="train_acc")
    plt.plot(baseline_hist_df["epoch"], baseline_hist_df["val_acc"], label="val_acc")
    plt.xlabel("epoch")
    plt.ylabel("accuracy (%)")
    plt.title("Baseline accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(baseline_hist_df["epoch"], baseline_hist_df["train_loss"], label="train_loss")
    plt.plot(baseline_hist_df["epoch"], baseline_hist_df["val_loss"], label="val_loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("Baseline loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 8. Optional Multi-Seed Baseline Runs

Jasmine notes did not clearly prove multi-seed averaging. This section lets you test whether seeds materially affect results.

Tip: set fewer epochs (for example 20-30) for quick variance checks, then rerun top seeds for full epochs.

In [ ]:
def run_multiseed_baseline(seeds=(42, 123, 2026), epochs=30):
    rows = []
    for s in seeds:
        print("\n" + "=" * 80)
        print(f"Running baseline seed={s}")
        print("=" * 80)
        _, _, info = run_baseline_once(seed=s, epochs=epochs)
        rows.append({
            "seed": s,
            "best_val_acc": info["best_val_acc"],
            "test_acc": info["test_acc"],
            "best_epoch": info["best_epoch"],
            "elapsed_min": info["elapsed_sec"] / 60.0,
            "checkpoint_path": info["checkpoint_path"],
        })

    df = pd.DataFrame(rows)
    summary = pd.DataFrame({
        "metric": ["best_val_acc", "test_acc"],
        "mean": [df["best_val_acc"].mean(), df["test_acc"].mean()],
        "std": [df["best_val_acc"].std(ddof=1), df["test_acc"].std(ddof=1)],
    })

    Path("saved_models").mkdir(exist_ok=True, parents=True)
    df.to_csv("saved_models/multiseed_baseline_results.csv", index=False)
    summary.to_csv("saved_models/multiseed_baseline_summary.csv", index=False)

    print("\nPer-seed results")
    display(df)
    print("\nSummary")
    display(summary)
    return df, summary


RUN_MULTI_SEED = False
MULTI_SEEDS = (42, 123, 777)
MULTI_SEED_EPOCHS = 30

if RUN_MULTI_SEED:
    multiseed_df, multiseed_summary = run_multiseed_baseline(seeds=MULTI_SEEDS, epochs=MULTI_SEED_EPOCHS)

## 9. Build the Improved Architecture

This improved version stays under 1.5M params and does not use transfer learning.

Improvements used:
- Squeeze-and-Excitation (SE) channel attention in residual blocks.
- Slightly stronger regularization and augmentation.
- Class-weighted CE + label smoothing option.
- Optional early stopping.

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        z = self.pool(x).view(b, c)
        s = self.fc(z).view(b, c, 1, 1)
        return x * s


class ResidualBlockSE(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = DepthwiseSeparableConv(in_channels, out_channels, stride)
        self.conv2 = DepthwiseSeparableConv(out_channels, out_channels, stride=1)
        self.se = SEBlock(out_channels, reduction=8)
        self.relu = nn.ReLU6(inplace=True)

        self.skip = None
        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        residual = x if self.skip is None else self.skip(x)
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.se(out)
        out = out + residual
        out = self.relu(out)
        return out


class LiteCNNV2(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.init_conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True),
        )

        self.block1 = self._make_layer(32, 48, 2, stride=2)
        self.block2 = self._make_layer(48, 96, 2, stride=2)
        self.block3 = self._make_layer(96, 192, 3, stride=2)
        self.block4 = self._make_layer(192, 384, 2, stride=2)

        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(384, num_classes),
        )

    def _make_layer(self, in_channels, out_channels, num_blocks, stride):
        layers = [ResidualBlockSE(in_channels, out_channels, stride)]
        for _ in range(1, num_blocks):
            layers.append(ResidualBlockSE(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.init_conv(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.global_avg_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


def compute_class_weights(train_dataset):
    labels = np.array(train_dataset.labels)
    counts = np.bincount(labels, minlength=len(EMOTIONS)).astype(np.float32)
    counts[counts == 0] = 1.0
    inv = 1.0 / counts
    weights = inv / inv.sum() * len(EMOTIONS)
    return torch.tensor(weights, dtype=torch.float32)


train_tfms_v2 = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(12),
    transforms.RandomAffine(degrees=0, translate=(0.12, 0.12), scale=(0.92, 1.08)),
    transforms.ColorJitter(brightness=0.25, contrast=0.25),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.12), ratio=(0.3, 3.3), value=0.0),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

v2_model = LiteCNNV2(num_classes=7)
v2_params = count_parameters(v2_model)
print(f"LiteCNNV2 params: {v2_params:,}")
print(f"Under 1.5M params? {'YES' if v2_params < 1_500_000 else 'NO'}")

## 10. Train the Improved Model and Save the Best Model

In [ ]:
def run_v2_once(seed=42, epochs=60):
    cfg = TrainConfig(
        epochs=epochs,
        batch_size=64,
        lr=1e-3,
        weight_decay=5e-5,
        val_fraction=0.1,
        split_seed=seed,
        run_seed=seed,
        num_workers=2,
        early_stopping_patience=12,
    )

    train_loader, val_loader, test_loader = make_loaders(
        data_root=DATA_ROOT,
        batch_size=cfg.batch_size,
        val_fraction=cfg.val_fraction,
        split_seed=cfg.split_seed,
        train_tfms=train_tfms_v2,
        test_tfms_local=test_tfms,
        num_workers=cfg.num_workers,
        pin_memory=True,
    )

    model = LiteCNNV2(num_classes=7)
    class_weights = compute_class_weights(train_loader.dataset)
    ckpt = Path("saved_models") / "LiteCNN_v2_best.pth"

    model, history, info = train_model(
        model,
        train_loader,
        val_loader,
        cfg,
        checkpoint_path=ckpt,
        class_weights=class_weights,
        label_smoothing=0.05,
    )

    test_acc = evaluate_accuracy(model, test_loader, DEVICE)
    info["test_acc"] = test_acc

    hist_df = pd.DataFrame(history)
    print("\nV2 summary")
    print(info)
    return model, hist_df, info


RUN_V2_NOW = True
V2_EPOCHS = 120

if RUN_V2_NOW:
    v2_model, v2_hist_df, v2_info = run_v2_once(seed=BASELINE_SEED, epochs=V2_EPOCHS)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(v2_hist_df["epoch"], v2_hist_df["train_acc"], label="train_acc")
    plt.plot(v2_hist_df["epoch"], v2_hist_df["val_acc"], label="val_acc")
    plt.xlabel("epoch")
    plt.ylabel("accuracy (%)")
    plt.title("V2 accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(v2_hist_df["epoch"], v2_hist_df["train_loss"], label="train_loss")
    plt.plot(v2_hist_df["epoch"], v2_hist_df["val_loss"], label="val_loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("V2 loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 11. Compare Baseline vs Improved Results

This final section creates a side-by-side summary and stores it to disk for report/presentation use.

In [ ]:
comparison_rows = []

if "baseline_info" in globals():
    comparison_rows.append({
        "model": "LiteCNNBaseline",
        "params": int(count_parameters(LiteCNNBaseline(num_classes=7))),
        "best_val_acc": float(baseline_info["best_val_acc"]),
        "test_acc": float(baseline_info["test_acc"]),
        "best_epoch": int(baseline_info["best_epoch"]),
        "elapsed_min": float(baseline_info["elapsed_sec"] / 60.0),
        "checkpoint": baseline_info["checkpoint_path"],
    })

if "v2_info" in globals():
    comparison_rows.append({
        "model": "LiteCNNV2",
        "params": int(count_parameters(LiteCNNV2(num_classes=7))),
        "best_val_acc": float(v2_info["best_val_acc"]),
        "test_acc": float(v2_info["test_acc"]),
        "best_epoch": int(v2_info["best_epoch"]),
        "elapsed_min": float(v2_info["elapsed_sec"] / 60.0),
        "checkpoint": v2_info["checkpoint_path"],
    })

if not comparison_rows:
    print("No results found yet. Run baseline and/or v2 training cells first.")
else:
    comp_df = pd.DataFrame(comparison_rows)
    comp_df["under_1_5M_params"] = comp_df["params"] < 1_500_000
    display(comp_df)

    Path("saved_models").mkdir(parents=True, exist_ok=True)
    comp_df.to_csv("saved_models/baseline_vs_v2_summary.csv", index=False)
    print("Saved summary to saved_models/baseline_vs_v2_summary.csv")

    plt.figure(figsize=(7, 4))
    plt.bar(comp_df["model"], comp_df["test_acc"])
    plt.ylabel("test accuracy (%)")
    plt.title("Baseline vs V2 test accuracy")
    plt.ylim(0, max(75, float(comp_df["test_acc"].max()) + 5))
    plt.grid(axis="y", alpha=0.25)
    plt.show()